# Preprocessing version 2
Preprocessing feels somewhat arbitrary in Scanpy compared to R. Here I will try to follow the tutorial from scirpy (TCR analysis) to merge transcriptomic and VDJ information into one object

In [ ]:
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
import scirpy as ir
import seaborn as sb
import scrublet as scr
from matplotlib import cm as mpl_cm
from matplotlib import pyplot as plt
from muon import prot as pt
from scipy.stats import median_abs_deviation

sc.set_figure_params(figsize=(4, 4))
sc.settings.verbosity = 2  # verbosity: errors (0), warnings (1), info (2), hints (3)

## Functions

In [ ]:
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRAC', 'TRB', 'TRDV', 'TRDC', 'TRG']

def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

def filter_tcr_genes(adata):
    non_tcr_genes = adata.var_names
    for prefix in tcr_gene_prefixs:
        non_tcr_genes = [el for el in non_tcr_genes if not el.startswith(prefix)]
    adata = adata[:, non_tcr_genes]
    return adata


def filter_high_var(adata, n):
    sc.pp.highly_variable_genes(adata, n_top_genes=n)
    adata = adata[:, adata.var['highly_variable']].copy()
    return adata


def calculate_umap(adata, n_high_var=5000, remove_tcr_genes=True):
    """
    Calculates UMAP coordinates
    :param adata: AnnData object
    :param n_high_var: how many variable genes to use, None uses raw data
    :param remove_tcr_genes: whether to remove TCR genes
    :return: results stored in adata object
    """
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)            
    sc.tl.umap(adata_tmp)
    adata.obsm['X_umap'] = adata_tmp.obsm['X_umap']


def calculate_leiden(adata, resolution, n_high_var=5000, remove_tcr_genes=True):
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)
    sc.tl.leiden(adata_tmp, resolution=resolution)
    adata.obs['leiden'] = adata_tmp.obs['leiden']

# Loading data

In [ ]:
path_data = "..\\..\\02_raw_data"
path_gex = f"{path_data}\\runs\\NK12_1_EBV\\count\\sample_filtered_feature_bc_matrix.h5"
path_tcr = f"{path_data}\\runs\\NK12_1_EBV\\vdj_t\\filtered_contig_annotations.csv"

First we load a conventional AnnData object with GEX information. We seperately load an AnnData object with TCR information and merge the two objects together into a MuData object (MultiOmicsData)

In [ ]:
## Gex
data = sc.read_10x_h5(path_gex, gex_only=False)
data.var_names_make_unique() #So that gene names are unique

## VDJ
data_tcr = ir.io.read_10x_vdj(path_tcr)


In [ ]:
data.var["feature_types"].value_counts()

### Gathering 3 types of data into a multi-omics object
Preprocessing functions came from here: https://scanpy-tutorials.readthedocs.io/en/multiomics/cite-seq/pbmc5k.html

In [ ]:
protein = data[:, data.var["feature_types"] == "Antibody Capture"].copy()

# I decided to run han solo as Mimi does it, but keep the visualization aspects of dsb
# rna = data[:, data.var["feature_types"] == "Gene Expression"].copy()

#Define a list of protein features
hashtags = ["P14", "P17", "P18", "P19", "P20", "P21", "GS_T", "GS_all"]
cite_seqs = ['CD4_adt', 'CD8_adt', "CD45RA_adt", "CD62L_adt", "CD95_adt"]
feature_barcode_ids = hashtags + cite_seqs

# Hashtag data
# scipy has recently removed the ".A" shorthand for ".toarray()"
data.uns['hashtags'] = hashtags
for h in hashtags:
    data.obs[h] = data[:, h].X.toarray().copy()

# CiteSeq Data
data.uns['cite_ids'] = cite_seqs
for c in cite_seqs:
    data.obs[c] = data[:, c].X.toarray().copy()

# Remove Barcodes from counts
data = data[:, [gene for gene in data.var_names if gene not in feature_barcode_ids]]
data.obs['sample'] = f'NEW1'

rna = data

In [ ]:
mdata = mu.MuData({"gex": data, "airr": data_tcr, "prot": protein})

# Update 2025_7 - save the counts into a separate layer. 
mdata["gex"].layers["counts"] = mdata["gex"].X.copy()

mdata

Define a variable "prot" that acts as a shorthand for mdata["prot"]

In [ ]:
prot = mdata.mod['prot']
prot

We also need to provide the RAW droplet data for normalization for empty droplets

In [ ]:
raw_NK = f"{path_data}\\runs\\NK12_1_EBV\\raw_feature_bc_matrix.h5"
## raw data
raw_dat = sc.read_10x_h5(raw_NK, gex_only=False)
raw_dat.var_names_make_unique() #So that gene names are unique

protein_raw = raw_dat[:, raw_dat.var["feature_types"] == "Antibody Capture"].copy()
rna_raw = raw_dat[:, raw_dat.var["feature_types"] == "Gene Expression"].copy()
mdata_raw = mu.MuData({"gex": rna_raw, "prot": protein_raw})


In [ ]:
#Calculates log10umi for each droplet in the raw data file to define where the "empty droplets" begin. 
mdata_raw['gex'].obs["log10umi"] = np.array(np.log10(mdata_raw['gex'].X.sum(axis=1) + 1)).reshape(-1)
mu.pl.histogram(mdata_raw['gex'], ['log10umi'], bins=50)
mu.pl.histogram(mdata_raw['gex'][mdata_raw['gex'].obs.log10umi >= 1], ['log10umi'], bins=50) #zoom in close

What I see here is a large peak below log10umi = 1

## Hash solo to discover doublets and demultiplex HTOs
Since this works on raw data, we want to preprocess this before removing doublets due to QC

### CD4 vs CD8 (This works well to predict CD4/CD8 doublets)

In [ ]:
CITEseqs = ["CD4_adt", "CD8_adt"]

def CITE_solo_by_sample(CITE_cols, col_name, n_noise_barcodes):
    mdata["gex"].obs[col_name] = 'NaN'

    dfs_donor = []
    mdata["gex"].obs = mdata["gex"].obs.drop(col_name, axis=1)
    sc.external.pp.hashsolo(mdata["gex"], CITE_cols, number_of_noise_barcodes=n_noise_barcodes)
    mdata["gex"].obs = mdata["gex"].obs.rename(columns={'Classification': col_name})

CITE_solo_by_sample(CITEseqs, 'cd4cd8_solo', n_noise_barcodes=1)  #n_noise_barcodes must be at least 1 less than the total number of barcodes present. 

#Transfer labels to prot too
mdata["prot"].obs["cd4cd8_solo"] = mdata["gex"].obs["cd4cd8_solo"].copy()
mdata["gex"].obs['cd4cd8_solo'].value_counts()
mdata.update()

### HTO demultiplexing

In [ ]:
def distributions_over_columns(mdata, columns, n_rows, n_cols, y_lim=0.003, x_lim=None, title=None):
    assert len(columns) <= n_rows * n_cols, f'n_rows*n_cols={n_rows*n_cols} does not fit {len(columns)} classes.'
    fig, axes = plt.subplots(ncols=n_cols, nrows=n_rows, figsize=(n_cols * 3, n_rows * 3))
    if type(axes) == np.ndarray:
        axes = axes.reshape(-1)
    else:
        axes = [axes]

    for i, ht in enumerate(columns):
        if np.sum(np.isnan(mdata.obs[ht])) != len(mdata):
            sb.distplot(mdata[mdata.obs[ht] >= 0].obs[ht], ax=axes[i])
        axes[i].set_ylim([0, y_lim])
        if x_lim is not None and x_lim < max(mdata.obs[ht]):
            axes[i].set_xlim([0, x_lim])
        axes[i].set_title(ht)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

distributions_over_columns(mdata["gex"], hashtags, 2, 4)

In [ ]:
def hash_solo_by_sample(hashtag_cols, col_name, n_noise_barcodes):
    mdata["gex"].obs[col_name] = 'NaN'

    dfs_donor = []
    mdata["gex"].obs = mdata["gex"].obs.drop(col_name, axis=1)
    sc.external.pp.hashsolo(mdata["gex"], hashtag_cols, number_of_noise_barcodes=n_noise_barcodes)
    mdata["gex"].obs = mdata["gex"].obs.rename(columns={'Classification': col_name})

hash_solo_by_sample(hashtags, 'pool', n_noise_barcodes=6)  #n_noise_barcodes must be at least 1 less than the total number of barcodes present. 
mdata["gex"].obs['pool'].value_counts()
mdata.update()

In [ ]:
for h in hashtags:
    mdata["gex"].obs[f'log_{h}'] = np.log(mdata["gex"].obs[h].values+1)
sb.violinplot(data=mdata["gex"].obs[[f'log_{h}' for h in hashtags]], density_norm='area')

## Normalizing protein data
... by using information for negative droplets

Preserve original counts in a layer before the normalisation:

In [ ]:
# First save the raw count data to a new layer
prot.layers['counts'] = prot.X

See here for details on dsb / Muon workflow: https://github.com/scverse/muon-tutorials/blob/master/cite-seq/1-CITE-seq-PBMC-5k.ipynb and https://github.com/niaid/dsb

In [ ]:
pt.pp.dsb(mdata, mdata_raw, empty_counts_range=(1,1), random_state=1)

In [ ]:
sc.pl.scatter(mdata['prot'], x="P14", y="P17", layers="counts")
sc.pl.scatter(mdata['prot'], x="P14", y="P17")

## Scrublet to find possible doublets - score will be used later

In [ ]:
#Define a view variable for the count matrix
counts_matrix = mdata["gex"].layers["counts"]

In [ ]:
## I have 19482    cells
mdata["gex"]

In [ ]:
## Anticipated doublet rate is aproximately given by ""Multiplet rate = 0.0008 * cells_recovered"" according to 10x Genomics tables. 
0.0008*18417

In [ ]:
scrub = scr.Scrublet(counts_matrix, expected_doublet_rate=0.147336)

In [ ]:
#This is the default pipeline
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30)

* Surprisingly, scrublet never assigns doublet score likelihoods above 0.5 - the example online clear states that it is supposed to be a bimodal distribution

In [ ]:
scrub.plot_histogram();

In [ ]:
fig, ax = scrub.plot_histogram()

fig.savefig(
    "../../05_plots/02_Overview_and_doublet_analysis/16_scrublet_NK12_1.pdf",
    bbox_inches="tight",
    format="pdf",
    dpi=300
)

### Return the doublet scores to the dataframe

In [ ]:
## Get annotation list
# Doublet scores per cell
scores = scrub.doublet_scores_obs_

## Use the bolean list to annotate csf cells
mdata["gex"].obs["scrublet_score"] = scores 

## QC and thresholding (MAD)
Export dataframe after rudimentary QC (MT, exclusion of empty droplets and dying cells) but BEFORE exclusion of doublets

In [ ]:
#Annotate QC metrics but do not filter
mdata["gex"].var['mt'] = mdata["gex"].var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(mdata["gex"], qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
#All outliers across the three metrics
mdata["gex"].obs["outlier"] = (
    is_outlier(mdata["gex"], "total_counts", 5) |
    is_outlier(mdata["gex"], "n_genes_by_counts", 5) |
    is_outlier(mdata["gex"], "pct_counts_mt", 5)
).astype("category")
mdata["gex"].obs.outlier.value_counts()

In [ ]:
# Plot the QC metrics, and decide a cutoff %%%%%%%%%%%%
# Number of Axes & plot size
ncols = 3
nrows = 1
figsize = 4
wspace = 1

fig, axs = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * figsize + figsize * wspace * (ncols - 1), nrows * figsize),
)


# Plot individual values  
sc.pl.violin(
    mdata["gex"],
    keys="n_genes_by_counts",
    groupby='outlier',
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[0]
)
sc.pl.violin(
    mdata["gex"],
    keys="total_counts",
    groupby="outlier",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[1]
)
sc.pl.violin(
    mdata["gex"],
    keys="pct_counts_mt",
    groupby="outlier",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[2]
)

In [ ]:
# Plot the QC metrics, and decide a cutoff %%%%%%%%%%%%
# Number of Axes & plot size
ncols = 3
nrows = 1
figsize = 4
wspace = 1

fig, axs = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(ncols * figsize + figsize * wspace * (ncols - 1), nrows * figsize),
)

# Plot individual values  
sc.pl.violin(
    mdata["gex"], 
    keys="n_genes_by_counts",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[0]
)
sc.pl.violin(
    mdata["gex"], 
    keys="total_counts",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[1]
)
sc.pl.violin(
    mdata["gex"], 
    keys="pct_counts_mt",
    jitter=0.4,
    multi_panel=False,
    show=False,
    ax=axs[2]
)

#n_genes_total (i.e. number of genes per cell)
axs[0].axhline(y=200, linewidth = 1.5, color = "black", linestyle = "--")

#number of total reads 
axs[1].axhline(y=400, linewidth = 1.5, color = "black", linestyle = "--")

#mitochondrial gene expression
axs[2].axhline(y=8, linewidth = 1.5, color = "black", linestyle = "--")

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Update 2025_4
# Genes needs to be detected in at least 3 cells. 

# A general lower limit on the number of genes detected per cell
# CSF = 200

# A general lower limit on the total_reads detected per cell
# CSF = 400


# %%% Filter functions from muon. 
## filter_var indicates filtering on genes
mu.pp.filter_var(mdata["gex"], 'n_cells_by_counts', lambda x: x >= 3)

## filter_obs indicates filtering on observations (i.e. cells)
mu.pp.filter_obs(mdata["gex"], 'n_genes_by_counts', lambda x: (x >= 200)) 
mu.pp.filter_obs(mdata["gex"], 'total_counts', lambda x: (x > 400))
mu.pp.filter_obs(mdata["gex"], 'pct_counts_mt', lambda x: x < 10)

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Normalization steps %%%%%%%%%%%%%%%%
sc.pp.normalize_total(mdata["gex"])
sc.pp.log1p(mdata["gex"])
sc.pp.highly_variable_genes(mdata["gex"], n_top_genes=5000)
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

#TCR steps %%%%%%%%%%%%%%%%%%%%%%%%%%%
ir.pp.index_chains(mdata)
ir.tl.chain_qc(mdata)    #Chain QC
mdata.obs['airr:chain_pairing'].value_counts()
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

#Update
mdata.update()

#Define NA clonotypes as "No IR detected"
mdata.obs['airr:chain_pairing'] = mdata.obs['airr:chain_pairing'].fillna('no_IR')

#Transfer the chain_pairing information
mdata["gex"].obs['chain_pairing'] = mdata["airr"].obs['chain_pairing']

With the neighbourhood graph computed, we can now perform clustering. We will use leiden clustering as an example.

In [ ]:
calculate_leiden(mdata["gex"], resolution=1, n_high_var=5000, remove_tcr_genes=True)
calculate_umap(mdata["gex"], n_high_var=5000, remove_tcr_genes=True)

In [ ]:
sc.pl.umap(mdata["gex"], color=["leiden", "scrublet_score", "total_counts", "outlier"], legend_loc="on data", legend_fontweight= 'bold', frameon= True , vmin=0, legend_fontoutline=2, cmap="RdBu_r")
sc.pl.umap(mdata["gex"], color=["chain_pairing", "CD3E", "CD14", "CD4"], legend_loc="on data", legend_fontweight= 'bold', frameon= True, size=40, vmin=0, legend_fontoutline=2, cmap="RdBu_r")
sc.pl.umap(mdata["gex"], color=["CD8A", "IL32", "TYROBP", "GZMK"], legend_loc="on data", legend_fontweight= 'bold', frameon= True, size=40, vmin=0, legend_fontoutline=2, cmap="RdBu_r")

### Double check the performance of hash_solo

In [ ]:
mdata.update()

In [ ]:
prot = mdata.mod['prot']
prot

In [ ]:
sc.pl.umap(mdata["gex"], color='pool')

In [ ]:
mdata["prot"].obs["pool"] = mdata["gex"].obs["pool"].copy() 
sc.pl.scatter(mdata['prot'], x="P14", y="P17", color="pool")

## Remove hashtag doublets and negatives

In [ ]:
#Remove observable doublets
print('Before Filtering Cluster: ', len(rna))
element_remove = ['Negative', "Doublet"]
mu.pp.filter_obs(rna, 'pool', lambda x: (x != "Doublet") & (x != "Negative"))
print('After Filtering Cluster: ', len(rna))

Inspect observable and unobservable duoblets with double TCR information

In [ ]:
mdata["gex"].obs['chain_pairing'] = mdata["airr"].obs['chain_pairing']
print(mdata["airr"].obs['chain_pairing'].value_counts())
print(mdata["gex"].obs['chain_pairing'].value_counts())

two "full chains" dropped markedly by removing observable duoblets. Let's remove the remaining "two-full chain" cells

In [ ]:
#Remove assumed unobservable doublets
print("Before filtering: ", mdata.obs['airr:chain_pairing'].value_counts())
# mu.pp.filter_obs(mdata, "airr:chain_pairing", lambda x: x != "two full chains") #Let's keep these in
print("After filtering: ", mdata.obs['airr:chain_pairing'].value_counts())

In [ ]:
sc.pl.umap(mdata["gex"], color='pool')

## CITEseq annotation
Because each sample will have variability in the CITEseq signal, we annotate each sample individually. 

In [ ]:
sc.pl.scatter(mdata['prot'], x="CD4_adt", y="CD8_adt", size=10, color="cd4cd8_solo")

In [ ]:
# Transfer the label and look at genes per cell
mdata["gex"].obs["cd4cd8_solo"] = mdata["prot"].obs["cd4cd8_solo"].copy()

df = mdata["gex"].obs[["cd4cd8_solo", "total_counts", "n_genes_by_counts"]]
plt.figure(figsize=(6, 4))

sb.boxplot(
    data=df,
    x="cd4cd8_solo",
    y="total_counts",
    showfliers=False  # hide outliers (optional)
)

sb.stripplot(
    data=df,
    x="cd4cd8_solo",
    y="total_counts",
    color="black",
    size=3,
    alpha=0.6,
    jitter=True
)

plt.xlabel("")
plt.ylabel("Total gene counts")
plt.tight_layout()
plt.show()

In [ ]:
# Transfer the label and look at genes per cell
mdata["gex"].obs["cd4cd8_solo"] = mdata["prot"].obs["cd4cd8_solo"].copy()

df = mdata["gex"].obs[["cd4cd8_solo", "total_counts", "n_genes_by_counts"]]
plt.figure(figsize=(6, 4))

sb.boxplot(
    data=df,
    x="cd4cd8_solo",
    y="n_genes_by_counts",
    showfliers=False  # hide outliers (optional)
)

sb.stripplot(
    data=df,
    x="cd4cd8_solo",
    y="n_genes_by_counts",
    color="black",
    size=3,
    alpha=0.6,
    jitter=True
)

plt.xlabel("")
plt.ylabel("n_genes_per_cell")
plt.tight_layout()
plt.show()

### CD4 & CD8 assignment (manual)

In [ ]:
prot = mdata["prot"]

# Define thresholds'
CD4_threshold = 10
CD8_threshold = 8

# Get column indices for the ADTs
i_cd4= prot.var_names.get_loc("CD4_adt")
i_cd8  = prot.var_names.get_loc("CD8_adt")

df = pd.DataFrame({
    "CD4_adt": prot.X[:, i_cd4].A1 if hasattr(prot.X, "A1") else prot.X[:, i_cd4],
    "CD8_adt":  prot.X[:, i_cd8].A1  if hasattr(prot.X, "A1") else prot.X[:, i_cd8],
    "cd4cd8_solo": prot.obs["cd4cd8_solo"].values,
    "pool": prot.obs["pool"].values
})

g = sb.FacetGrid(
    df,
    col="pool",
    height=5,
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sb.kdeplot,
    x="CD4_adt",
    y="CD8_adt",
    fill=True,
    thresh=0.05
)

# Add gating lines to all panels
for ax in g.axes.flat:
    ax.axvline(CD4_threshold, linestyle="--", linewidth=1, color="black")
    ax.axhline(CD8_threshold, linestyle="--", linewidth=1, color="black")

# Turn grid off
for ax in g.axes.flat:
    ax.grid(False)

g.set_axis_labels("CD4 ADT", "CD8 ADT")
plt.tight_layout()
plt.show()

In [ ]:
### CD8 annotation
i = mdata["prot"].var_names.get_loc("CD8_adt")                #Finds the index where CD95_adt values are. 
x = np.asarray(mdata["prot"].X[:, i]).ravel()                 #Extracts the value into a 1 dimensional array
mdata["prot"].obs["CD8_high"] = x >= CD8_threshold          #Checks if the array position is above a threshold - gives values 0 and 1 (1 = above)    THRESHOLD DEFINED IN THE PLOTS ABOVE

sc.pl.scatter(mdata['prot'], x="CD4_adt", y="CD8_adt", size=10, color="CD8_high")  #Works

In [ ]:
### CD4 annotation
i = mdata["prot"].var_names.get_loc("CD4_adt")                #Finds the index where CD95_adt values are. 
x = np.asarray(mdata["prot"].X[:, i]).ravel()                 #Extracts the value into a 1 dimensional array
mdata["prot"].obs["CD4_high"] = x >= CD4_threshold          #Checks if the array position is above a threshold - gives values 0 and 1 (1 = above)    THRESHOLD DEFINED IN THE PLOTS ABOVE

sc.pl.scatter(mdata['prot'], x="CD4_adt", y="CD8_adt", size=10, color="CD4_high")  #Works

In [ ]:
mdata["prot"].obs["cd4cd8_cs_assignment"] = pd.Categorical(
    np.where((mdata["prot"].obs["CD4_high"] == 0) & (mdata["prot"].obs["CD8_high"] == 0), "adt_negative", 
             np.where((mdata["prot"].obs["CD4_high"] == 1) & (mdata["prot"].obs["CD8_high"] == 0), "CD4", 
                      np.where((mdata["prot"].obs["CD4_high"] == 0) & (mdata["prot"].obs["CD8_high"] == 1), "CD8",
                               np.where((mdata["prot"].obs["CD4_high"] == 1) & (mdata["prot"].obs["CD8_high"] == 1), "CD4/CD8", "error")))))

sc.pl.scatter(mdata['prot'], x="CD4_adt", y="CD8_adt", size=10, color="cd4cd8_cs_assignment")  #Works

### Clean up

In [ ]:
###Clean-up.
#Make sure only the indexes recorded in GEX are used in the other modalities too
index_list = mdata["gex"].obs.index.tolist()

## define a cell index
mdata["airr"].obs["cell_index"] = mdata["airr"].obs.index
mdata["prot"].obs["cell_index"] = mdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that mdata["airr"].obs and mdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata["prot"], 'filter_index', lambda x: x == "GEX_present")

#Clean up categories
mdata["prot"].obs["pool"] = mdata["gex"].obs["pool"].copy()
mdata["prot"].obs["pool"] = mdata["prot"].obs["pool"].cat.remove_unused_categories()

### CD45RA vs CD62L (must be done manually)

In [ ]:
prot = mdata["prot"]

# Define thresholds
CD45_threshold = 16
CD62L_threshold = 16

# Get column indices for the ADTs
i_cd45ra = prot.var_names.get_loc("CD45RA_adt")
i_cd62l  = prot.var_names.get_loc("CD62L_adt")
i_cd95  = prot.var_names.get_loc("CD95_adt")

df = pd.DataFrame({
    "CD45RA_adt": prot.X[:, i_cd45ra].A1 if hasattr(prot.X, "A1") else prot.X[:, i_cd45ra],
    "CD62L_adt":  prot.X[:, i_cd62l].A1  if hasattr(prot.X, "A1") else prot.X[:, i_cd62l],
    "CD95_adt":  prot.X[:, i_cd95].A1  if hasattr(prot.X, "A1") else prot.X[:, i_cd95],
    "cd4cd8_cs_assignment": prot.obs["cd4cd8_cs_assignment"].values,
    "cd4cd8_solo": prot.obs["cd4cd8_solo"].values,
    "pool": prot.obs["pool"].values
})

g = sb.FacetGrid(
    df,
    col="cd4cd8_cs_assignment",
    row="pool",
    height=5,
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sb.kdeplot,
    x="CD45RA_adt",
    y="CD62L_adt",
    fill=True,
    thresh=0.05
)

# Add gating lines to all panels
for ax in g.axes.flat:
    ax.axvline(CD45_threshold, linestyle="--", linewidth=1, color="black")
    ax.axhline(CD62L_threshold, linestyle="--", linewidth=1, color="black")

# Turn grid off
for ax in g.axes.flat:
    ax.grid(False)

g.set_axis_labels("CD45RA ADT", "CD62L ADT")
plt.tight_layout()
plt.show()

In [ ]:
#Define threshold
CD95_threshold = 11.5

g = sb.FacetGrid(
    df,
    col="cd4cd8_solo",
    row="pool",
    height=5,
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sb.kdeplot,
    x="CD95_adt",
    y="CD62L_adt",
    fill=True,
    thresh=0.05
)

# Add gating lines to all panels
for ax in g.axes.flat:
    ax.axvline(CD95_threshold, linestyle="--", linewidth=1, color="black")
    
# Turn grid off
for ax in g.axes.flat:
    ax.grid(False)
    
plt.tight_layout()
plt.show()

### Annotations for CD45RA, CD62L, and CD95

In [ ]:
### CD95 first 
i = mdata["prot"].var_names.get_loc("CD95_adt")   #Finds the index where CD95_adt values are. 
x = np.asarray(mdata["prot"].X[:, i]).ravel()     #Extracts the value into a 1 dimensional array
mdata["prot"].obs["CD95_high"] = x >= CD95_threshold          #Checks if the array position is above a threshold - gives values 0 and 1 (1 = above)    THRESHOLD DEFINED IN THE PLOTS ABOVE

# sc.pl.scatter(mdata['prot'], x="CD62L_adt", y="CD95_adt", size=10, color="CD95_high")  #Works

In [ ]:
### CD45RA  
i = mdata["prot"].var_names.get_loc("CD45RA_adt")   #Finds the index where the adt values are. 
x = np.asarray(mdata["prot"].X[:, i]).ravel()       #Extracts the value into a 1 dimensional array
mdata["prot"].obs["CD45RA_high"] = x >= CD45_threshold          #Checks if the array position is above a threshold - gives values 0 and 1 (1 = above)   THRESHOLD DEFINED IN THE PLOTS ABOVE

# sc.pl.scatter(mdata['prot'], x="CD45RA_adt", y="CD62L_adt", size=10, color="CD45RA_high")  #Works

In [ ]:
### CD62L  
i = mdata["prot"].var_names.get_loc("CD62L_adt")   #Finds the index where the adt values are. 
x = np.asarray(mdata["prot"].X[:, i]).ravel()       #Extracts the value into a 1 dimensional array
mdata["prot"].obs["CD62L_high"] = x >= CD62L_threshold         #Checks if the array position is above a threshold - gives values 0 and 1 (1 = above)    THRESHOLD DEFINED IN THE PLOTS ABOVE

# sc.pl.scatter(mdata['prot'], x="CD45RA_adt", y="CD62L_adt", size=10, color="CD62L_high")  #Works

In [ ]:
mdata["prot"].obs["Differentiation"] = pd.Categorical(
    np.where((mdata["prot"].obs["CD62L_high"] == 0) & (mdata["prot"].obs["CD45RA_high"] == 0), "Tem", 
             np.where((mdata["prot"].obs["CD62L_high"] == 1) & (mdata["prot"].obs["CD45RA_high"] == 0), "Tcm", 
                      np.where((mdata["prot"].obs["CD62L_high"] == 0) & (mdata["prot"].obs["CD45RA_high"] == 1), "Temra",
                               np.where((mdata["prot"].obs["CD62L_high"] == 1) & (mdata["prot"].obs["CD45RA_high"] == 1) & (mdata["prot"].obs["CD95_high"] == 0), "Tn",
                                        np.where((mdata["prot"].obs["CD62L_high"] == 1) & (mdata["prot"].obs["CD45RA_high"] == 1) & (mdata["prot"].obs["CD95_high"] == 1), "Tscm", "error"))))))

sc.pl.scatter(mdata['prot'], x="CD45RA_adt", y="CD62L_adt", size=10, color="Differentiation")  #Works

In [ ]:
### Important to update to bring the differentiation back into global acces (for umaps later)
mdata.update()

In [ ]:
mdata_temp = mdata.copy()
mu.pp.filter_obs(mdata_temp["prot"], 'pool', lambda x: x == "P14")
mu.pp.filter_obs(mdata_temp["prot"], 'cd4cd8_solo', lambda x: x == "CD8_adt")
mdata_temp.update()

sc.pl.scatter(mdata_temp['prot'], x="CD45RA_adt", y="CD62L_adt", size=20, color="Differentiation")  #Works
sc.pl.scatter(mdata_temp['prot'], x="CD95_adt", y="CD62L_adt", size=20, color="Differentiation")  #Works

In [ ]:
mdata_temp = mdata.copy()
mu.pp.filter_obs(mdata_temp["prot"], 'pool', lambda x: x == "P17")
mu.pp.filter_obs(mdata_temp["prot"], 'cd4cd8_solo', lambda x: x == "CD8_adt")
mdata_temp.update()

sc.pl.scatter(mdata_temp['prot'], x="CD45RA_adt", y="CD62L_adt", size=20, color="Differentiation")  #Works

In [ ]:
mdata_temp = mdata.copy()
mu.pp.filter_obs(mdata_temp["prot"], 'pool', lambda x: x == "P18")
mu.pp.filter_obs(mdata_temp["prot"], 'cd4cd8_solo', lambda x: x == "CD8_adt")
mdata_temp.update()

sc.pl.scatter(mdata_temp['prot'], x="CD45RA_adt", y="CD62L_adt", size=20, color="Differentiation")  #Works

In [ ]:
mdata_temp = mdata.copy()
mu.pp.filter_obs(mdata_temp["prot"], 'pool', lambda x: x == "P19")
mu.pp.filter_obs(mdata_temp["prot"], 'cd4cd8_solo', lambda x: x == "CD8_adt")
mdata_temp.update()

sc.pl.scatter(mdata_temp['prot'], x="CD45RA_adt", y="CD62L_adt", size=20, color="Differentiation")  #Works

In [ ]:
mdata_temp = mdata.copy()
mu.pp.filter_obs(mdata_temp["prot"], 'pool', lambda x: x == "P20")
mu.pp.filter_obs(mdata_temp["prot"], 'cd4cd8_solo', lambda x: x == "CD8_adt")
mdata_temp.update()

sc.pl.scatter(mdata_temp['prot'], x="CD45RA_adt", y="CD62L_adt", size=10, color="Differentiation")  #Works

In [ ]:
mdata_temp = mdata.copy()
mu.pp.filter_obs(mdata_temp["prot"], 'pool', lambda x: x == "P21")
mu.pp.filter_obs(mdata_temp["prot"], 'cd4cd8_solo', lambda x: x == "CD8_adt")
mdata_temp.update()

sc.pl.scatter(mdata_temp['prot'], x="CD45RA_adt", y="CD62L_adt", size=20, color="Differentiation")  #Works

### CITEseq visual check

In [ ]:
mu.pl.embedding(mdata, basis="gex:umap", color=["prot:CD4_adt", "prot:CD8_adt", "prot:CD45RA_adt", "prot:CD62L_adt", "prot:CD95_adt"], legend_loc="on data", legend_fontweight= 'bold', frameon= True, size=40, vmin=0, legend_fontoutline=2, cmap="RdBu_r")


In [ ]:
mu.pl.embedding(mdata, basis="gex:umap", color=["prot:Differentiation"], legend_fontweight= 'bold', frameon= True, size=40, vmin=0, legend_fontoutline=2, cmap="RdBu_r")


## Save without the GS patient

In [ ]:
mu.pp.filter_obs(mdata["gex"], 'pool', lambda x: x != "GS_T")

In [ ]:
mdata.write(filename=r"C:\Users\nipag\coding\MS.schober\04_data_objects\01_preprocessed_scseq_scanpy\2025_12 - with EBV\NK_12_1.h5mu")